# 00 — Data Inventory

**What this notebook is for.** Before building anything, write down exactly what data
exists, how much of it is usable, and what is missing. The evaluator asked for a data
inventory (point 9) and a list of missing GIS datasets (point 10); this notebook produces
both, as a file you can hand to someone.

**What it does**

1. Confirms every raw file is present and reports its size.
2. Reads only the first and last rows of each file — enough to check the time range and
   detect truncation without loading 52 GB.
3. Counts stations per dataset per year, so roster growth is visible.
4. Writes `docs/data_inventory.md`.

**Runtime:** a few minutes. It never loads a whole file into memory.

In [1]:
import sys, pathlib
# Make the shared library importable no matter where Jupyter was started from.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from bkkflood import CFG, PATHS
print("project root:", ROOT)
print("config version:", CFG["project"]["version"])

project root: /Users/pritimmondal/Projects/bkk-flood-forecast
config version: 2.0.0


## 1. Do the files exist, and how big are they?

If a file is missing here, nothing downstream will work. Better to find out now than
forty minutes into an ingestion run.

In [2]:
import subprocess, os

rows = []
for ds in CFG["raw"]["datasets"]:
    for year in CFG["raw"]["years"]:
        path = PATHS.raw_csv(ds, year)
        rows.append({
            "dataset": ds,
            "year": year,
            "filename": path.name,
            "exists": path.exists(),
            "size_gb": round(path.stat().st_size / 1e9, 2) if path.exists() else 0.0,
        })

files = pd.DataFrame(rows)
print(f"Total raw size: {files['size_gb'].sum():.1f} GB")
print(f"Missing files:  {(~files['exists']).sum()}")
files.pivot_table(index="year", columns="dataset", values="size_gb", aggfunc="sum")

Total raw size: 54.9 GB
Missing files:  0


dataset,flood,flow,rain,water
year,,,,
2019,1.18,0.43,1.94,3.96
2020,1.18,0.43,1.94,3.97
2021,1.20,0.43,1.94,3.96
2022,1.22,0.43,1.96,4.07
2023,1.29,0.43,1.96,4.12
2024,1.29,0.43,1.96,4.72
2025,1.29,0.43,1.96,4.75


## 2. Time coverage — is any file truncated?

Each file should cover a full calendar year at 5-minute cadence, so the last timestamp
should be 31 December, 23:55. A file ending in, say, August means the delivery was cut
short and every model trained on it will be quietly biased against the late-year monsoon.

We read the head and tail with shell tools rather than pandas, because reading the last
line of a 5 GB CSV should not require loading the first 5 GB of it.

In [3]:
def first_and_last_line(path, encoding="utf-8-sig"):
    """Read the header, the first data row, and the last data row — cheaply."""
    with open(path, "r", encoding=encoding, errors="replace") as fh:
        header = fh.readline().rstrip("\n")
        first = fh.readline().rstrip("\n")
    # Seek to the end and walk backwards to find the final newline.
    with open(path, "rb") as fh:
        fh.seek(0, os.SEEK_END)
        size = fh.tell()
        block = min(8192, size)
        fh.seek(-block, os.SEEK_END)
        tail = fh.read().decode("utf-8", errors="replace").strip().split("\n")
    return header, first, tail[-1]


coverage = []
for _, r in files[files["exists"]].iterrows():
    path = PATHS.raw_csv(r["dataset"], r["year"])
    header, first, last = first_and_last_line(path)
    coverage.append({
        "dataset": r["dataset"], "year": r["year"],
        "columns": header.count(",") + 1,
        "first_row": first[:70],
        "last_row": last[:70],
    })

coverage = pd.DataFrame(coverage)
coverage

,dataset,year,columns,first_row,last_row
0,flood,2019,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2019-0...","FL.WTN.02,ถ.อโศกมนตรี ช่วงแยกชิโนไทย,2019-12-3..."
1,flood,2020,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2020-0...","FL.WTN.02,ถ.อโศกมนตรี ช่วงแยกชิโนไทย,2020-12-3..."
2,flood,2021,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2021-0...","FL.WTN.02,ถ.อโศกมนตรี ช่วงแยกชิโนไทย,2021-12-3..."
3,flood,2022,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2022-0...","FL.WTN.02,ถ.อโศกมนตรี ช่วงแยกชิโนไทย,2022-12-3..."
4,flood,2023,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2023-0...","FL.WTN.03,ถ.สุขุมวิท 71 ช่วงซอยปรีดีพนมยงค์ 20..."
5,flood,2024,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2024-0...","FL.WTN.03,ถ.สุขุมวิท 71 ช่วงซอยปรีดีพนมยงค์ 20..."
6,flood,2025,4,"FL.BBN.01,ถ.เอกชัย ช่วงห้างบิ๊กซีบางบอน,2025-0...","FL.WTN.03,ถ.สุขุมวิท 71 ช่วงซอยปรีดีพนมยงค์ 20..."
7,flow,2019,7,"FW.BBU.01,จุดวัดคลองบางบัว,2019-01-01 00:00:00...","FW.TWW.01,ประตูระบายน้ำคลองทวีวัฒนา,2019-12-31..."
8,flow,2020,7,"FW.BBU.01,จุดวัดคลองบางบัว,2020-01-01 00:00:00...","FW.TWW.01,ประตูระบายน้ำคลองทวีวัฒนา,2020-12-31..."
9,flow,2021,7,"FW.BBU.01,จุดวัดคลองบางบัว,2021-01-01 00:00:00...","FW.TWW.01,ประตูระบายน้ำคลองทวีวัฒนา,2021-12-31..."


**How to read this.** Look at `last_row`. Every one should show a December 31st
timestamp near 23:55. Anything else means that year is incomplete and should be flagged
in the inventory — and probably excluded from training, because a partial year distorts
seasonal features.

## 3. Station rosters

How many stations report in each dataset each year, and did the network grow? Growth is
normal — BMA adds sensors — but it matters for two reasons:

* a station that appears in 2024 has no history to learn from, and
* a station that *disappears* is either broken or decommissioned, and if you do not
  notice, you will keep expecting forecasts for a site that no longer exists.

Counting distinct stations does need a real pass over the code column, so this cell reads
one column at a time in chunks. Budget a few minutes.

In [4]:
roster = []
for ds, spec in CFG["raw"]["datasets"].items():
    for year in CFG["raw"]["years"]:
        path = PATHS.raw_csv(ds, year)
        if not path.exists():
            continue
        codes = set()
        for chunk in pd.read_csv(path, encoding=CFG["raw"]["encoding"],
                                 usecols=[spec["code_col"]], dtype=str,
                                 chunksize=2_000_000, quotechar='"'):
            codes.update(chunk[spec["code_col"]].dropna().unique())
        roster.append({"dataset": ds, "year": year, "stations": len(codes),
                       "codes": codes})

roster_df = pd.DataFrame(roster)
pivot = roster_df.pivot(index="year", columns="dataset", values="stations")
print(pivot.to_string())
pivot

dataset  flood  flow  rain  water
year                             
2019        99    30   130    255
2020        99    30   130    255
2021       102    30   130    255
2022       102    30   131    262
2023       107    30   131    265
2024       107    30   131    297
2025       107    30   131    300


dataset,flood,flow,rain,water
year,,,,
2019,99,30,130,255
2020,99,30,130,255
2021,102,30,130,255
2022,102,30,131,262
2023,107,30,131,265
2024,107,30,131,297
2025,107,30,131,300


In [5]:
# Which stations joined or left between the first and last year?
for ds in roster_df["dataset"].unique():
    sub = roster_df[roster_df["dataset"] == ds].sort_values("year")
    first, last = sub.iloc[0], sub.iloc[-1]
    added = sorted(last["codes"] - first["codes"])
    removed = sorted(first["codes"] - last["codes"])
    print(f"{ds:6s}  {first['year']}: {len(first['codes']):3d} -> "
          f"{last['year']}: {len(last['codes']):3d}   "
          f"+{len(added)} added, -{len(removed)} removed")
    if removed:
        print(f"         gone: {', '.join(removed[:8])}"
              + (" ..." if len(removed) > 8 else ""))

flood   2019:  99 -> 2025: 107   +8 added, -0 removed
flow    2019:  30 -> 2025:  30   +0 added, -0 removed
rain    2019: 130 -> 2025: 131   +1 added, -0 removed
water   2019: 255 -> 2025: 300   +45 added, -0 removed


## 4. The spatial join — can rain be matched to flood stations?

Station codes look like `FL.BKN.02`: a dataset prefix, a **district code**, and a number.
If a flood station's district code also appears among the rain stations, we can attach
that district's rainfall to it. This is our only spatial join until real coordinates
arrive.

Check it explicitly. If coverage were partial, some flood stations would silently receive
no rain features at all, and their forecasts would be far worse for a reason nobody would
think to look for.

In [6]:
def prefixes(ds):
    sub = roster_df[roster_df["dataset"] == ds]
    if sub.empty:
        return set()
    codes = set().union(*sub["codes"])
    return {c.split(".")[1] for c in codes if isinstance(c, str) and c.count(".") >= 2}

flood_p, rain_p = prefixes("flood"), prefixes("rain")
water_p, flow_p = prefixes("water"), prefixes("flow")

print(f"flood districts: {len(flood_p)}")
for name, other in [("rain", rain_p), ("water", water_p), ("flow", flow_p)]:
    matched = flood_p & other
    print(f"  {name:6s} covers {len(matched):3d}/{len(flood_p)} flood districts"
          f"   unmatched: {sorted(flood_p - other)[:6]}")

flood districts: 33
  rain   covers  33/33 flood districts   unmatched: []
  water  covers  13/33 flood districts   unmatched: ['BKE', 'BKM', 'BKN', 'BKP', 'BRK', 'DDG']
  flow   covers   3/33 flood districts   unmatched: ['BBN', 'BKE', 'BKL', 'BKM', 'BKN', 'BKP']


**Expected result.** Rain covers all of them. Water and flow cover only a handful,
because their codes are named after *canals*, not districts — `WL.STN.01` is a canal
initialism, not Sathon. That is why water and flow enter the model as citywide averages
rather than local values, and it is one of the strongest arguments for getting station
coordinates from BMA.

## 5. What we have, and what we are missing

The missing half of this table is the part to take to the BMA meeting. Each row says what
the dataset would unlock, so the ask is a capability, not a filename.

In [7]:
have = pd.DataFrame([
    ["Rainfall time series", "2019-2025, 5-min, ~131 gauges", "yes", "Primary flood driver"],
    ["Canal water level",    "2019-2025, 5-min, ~300 sites",  "yes", "Drainage system state"],
    ["Canal flow",           "2019-2025, 5-min, ~30 sites",   "yes", "Discharge and backflow"],
    ["Flood depth",          "2019-2025, 5-min, ~107 sites",  "yes", "The target variable"],
    ["Admin boundaries",     "GeoJSON + shapefile, 50 khet",  "yes", "Map choropleth, district rollup"],
    ["DEM / DTM",            "SRTM 31 m + 1 m DTM tiles",     "yes", "Terrain features (SRTM too coarse)"],
], columns=["dataset", "detail", "have", "used_for"])

missing = pd.DataFrame([
    ["Station coordinates", "critical",
     "Every water and flow sensor is currently a citywide average instead of a local "
     "input. Coordinates alone would let us join canals to the flood sites they drain."],
    ["Station metadata", "critical",
     "We do not know the datum of wl_in, the sensor model, or the installation height. "
     "Without the datum, absolute water levels are uninterpretable and we can only use "
     "changes."],
    ["Canal network topology", "high",
     "Which canal flows into which. Unlocks graph models and upstream-to-downstream "
     "routing, which is how flooding actually propagates."],
    ["Drainage and pump records", "high",
     "Pumping stations actively change the outcome. A model blind to pump operation is "
     "trying to predict a system while ignoring its controller."],
    ["Road-level elevation (LiDAR)", "high",
     "Street flooding happens in dips of 20-50 cm. A 31 m DEM cannot see them, which is "
     "why terrain features tested weak."],
    ["Radar rainfall (TMD)", "high",
     "Our rain is a district average; Bangkok floods from small intense cells. This is "
     "the single biggest expected accuracy gain."],
    ["Chao Phraya tide gauge", "medium",
     "High tide holds the drainage gates shut. We currently reconstruct tidal phase from "
     "astronomy, which gives timing but not height."],
    ["Historical flood reports", "medium",
     "Independent ground truth to validate the sensor-derived labels against."],
], columns=["dataset", "priority", "why_it_matters"])

print(have.to_string(index=False))
print()
print(missing[["dataset", "priority"]].to_string(index=False))

             dataset                        detail have                           used_for
Rainfall time series 2019-2025, 5-min, ~131 gauges  yes               Primary flood driver
   Canal water level  2019-2025, 5-min, ~300 sites  yes              Drainage system state
          Canal flow   2019-2025, 5-min, ~30 sites  yes             Discharge and backflow
         Flood depth  2019-2025, 5-min, ~107 sites  yes                The target variable
    Admin boundaries  GeoJSON + shapefile, 50 khet  yes    Map choropleth, district rollup
           DEM / DTM     SRTM 31 m + 1 m DTM tiles  yes Terrain features (SRTM too coarse)

                     dataset priority
         Station coordinates critical
            Station metadata critical
      Canal network topology     high
   Drainage and pump records     high
Road-level elevation (LiDAR)     high
        Radar rainfall (TMD)     high
      Chao Phraya tide gauge   medium
    Historical flood reports   medium


## 6. Write the inventory document

In [8]:
from datetime import date

lines = [
    "# Data Inventory — Bangkok Flood Forecast",
    "",
    f"Generated by `notebooks/00_data_inventory.ipynb` on {date.today().isoformat()}.",
    "",
    "## 1. Raw datasets held",
    "",
    f"Four sensor datasets, {min(CFG['raw']['years'])}-{max(CFG['raw']['years'])}, "
    f"5-minute cadence, **{files['size_gb'].sum():.1f} GB** in total.",
    "",
    have.to_markdown(index=False),
    "",
    "### File sizes by year (GB)",
    "",
    files.pivot_table(index="year", columns="dataset",
                      values="size_gb", aggfunc="sum").round(2).to_markdown(),
    "",
    "### Station counts by year",
    "",
    pivot.to_markdown(),
    "",
    "## 2. Known data quirks",
    "",
    "Each of these is real and verified, and each is handled in `src/bkkflood/ingest.py`.",
    "",
    "| Quirk | Effect if ignored | Handling |",
    "|---|---|---|",
    "| UTF-8 BOM on every header | first column name is corrupted | read with `utf-8-sig` |",
    "| Missing values are the text `NULL` | column parses as string, not number | coerce to NaN |",
    "| Rain filenames inconsistent (`2019.csv` vs `Rain 2021.csv`) | file not found | per-year filename rule |",
    "| Water station names contain commas | naive splitting corrupts rows | quoted CSV parser |",
    "| `FW.PKG.01` reads +/-3300 m3/s | swamps every canal average | excluded from features (river gauge, not a fault) |",
    "| Rain accumulations are gap-filled by BMA | a filled value looks like a measurement | flagged with `gapfilled` |",
    "| Implausible values (762 mm/24 h) | poisons scaling and thresholds | range-checked, nulled, flagged |",
    "",
    "## 3. The spatial join",
    "",
    f"- Rain covers **{len(flood_p & rain_p)}/{len(flood_p)}** flood districts by code prefix "
    "-> rainfall is joined per district.",
    f"- Water covers **{len(flood_p & water_p)}/{len(flood_p)}**, flow **{len(flood_p & flow_p)}/{len(flood_p)}** "
    "-> their codes are canal initialisms, not districts, so both enter the model as "
    "citywide aggregates.",
    "",
    "This is the strongest practical argument for obtaining station coordinates.",
    "",
    "## 4. Datasets we do not have",
    "",
    missing.to_markdown(index=False),
    "",
    "## 5. What this means for the model",
    "",
    "- Rainfall is a **district average**. Bangkok floods from localised convective "
    "cells, so the average smooths away the peak that causes the flood. Radar would "
    "address this directly.",
    "- Water and canal flow are **citywide**, so the model can tell that the network as "
    "a whole is under stress but not which canal is failing.",
    "- Terrain is **31 m SRTM**, which cannot resolve the street-level dips where urban "
    "flooding actually collects.",
    "- Flood labels come from **sensors only**. Any flood at a location without a sensor "
    "is invisible to both the model and the evaluation.",
]

out_path = PATHS.docs / "data_inventory.md"
out_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {out_path}")
print("\n".join(lines[:30]))

Wrote /Users/pritimmondal/Projects/bkk-flood-forecast/docs/data_inventory.md
# Data Inventory — Bangkok Flood Forecast

Generated by `notebooks/00_data_inventory.ipynb` on 2026-07-29.

## 1. Raw datasets held

Four sensor datasets, 2019-2025, 5-minute cadence, **54.9 GB** in total.

| dataset              | detail                        | have   | used_for                           |
|:---------------------|:------------------------------|:-------|:-----------------------------------|
| Rainfall time series | 2019-2025, 5-min, ~131 gauges | yes    | Primary flood driver               |
| Canal water level    | 2019-2025, 5-min, ~300 sites  | yes    | Drainage system state              |
| Canal flow           | 2019-2025, 5-min, ~30 sites   | yes    | Discharge and backflow             |
| Flood depth          | 2019-2025, 5-min, ~107 sites  | yes    | The target variable                |
| Admin boundaries     | GeoJSON + shapefile, 50 khet  | yes    | Map choropleth, district rollup 